## Environment Setup

Install all required packages for the RAG pipeline: LangChain for orchestration, MLflow for model tracking, and Databricks Vector Search for retrieval. The kernel restarts after installation to load the new libraries.

## Document Ingestion & Parsing

This section reads raw documents (PDFs) from the Unity Catalog Volume, parses them using `ai_parse_document()`, extracts text content, and loads it into a Delta table (`jobs_knowledge_base`) that serves as the source for Vector Search indexing.

## Knowledge Base Storage

Create the Delta table with Change Data Feed enabled (required for Vector Search sync), then insert the parsed document content. After this step, go to the Catalog UI to sync the table with the Vector Search index.

In [0]:
%pip install -qqq -U databricks-sdk databricks-langchain databricks-vectorsearch langsmith>=0.1.125 langchain==0.3.27 mlflow-skinny[databricks]>=3.4.0 

dbutils.library.restartPython()

In [0]:
%sql
SELECT path FROM READ_FILES('/Volumes/isa632_7474656346303369/default/isa632_boopatt/project/', format => 'binaryFile')

In [0]:
%sql
    
-- ai_parse_document is available in DBR 17.1 or serverless runtime
SELECT ai_parse_document(content) AS parsed_document
  FROM READ_FILES('/Volumes/isa632_7474656346303369/default/isa632_boopatt/project/', format => 'binaryFile') 

In [0]:
%sql
    
SELECT array_join(
            transform(parsed_document:document.elements::ARRAY<STRUCT<content:STRING>>, x -> x.content), '\n') AS content,
           path as doc_uri
    FROM (
      SELECT ai_parse_document(content) AS parsed_document, path
      FROM READ_FILES('/Volumes/isa632_7474656346303369/default/isa632_boopatt/project/', format => 'binaryFile') 
       -- ADDED FIX LIMIT FOR DEMO COST - DROP IT IN REAL WORKLOAD
    )

In [0]:
%sql
    
CREATE TABLE IF NOT EXISTS isa632_7474656346303369.boopatt.jobs_knowledge_base (
  id BIGINT GENERATED ALWAYS AS IDENTITY,
  content STRING,
  doc_uri STRING)
  TBLPROPERTIES (delta.enableChangeDataFeed = true);


# Insert parsed documents into knowledge base



In [0]:
%sql
    
INSERT OVERWRITE TABLE isa632_7474656346303369.boopatt.jobs_knowledge_base ( content, doc_uri)
SELECT content, doc_uri
FROM (
  SELECT
    content,
    doc_uri
  FROM (
    SELECT array_join(
            transform(parsed_document:document.elements::ARRAY<STRUCT<content:STRING>>, x -> x.content), '\n') AS content,
           path as doc_uri
    FROM (
      SELECT ai_parse_document(content) AS parsed_document, path
      FROM READ_FILES('/Volumes/isa632_7474656346303369/default/isa632_boopatt/project/', format => 'binaryFile') 
       -- ADDED FIX LIMIT FOR DEMO COST - DROP IT IN REAL WORKLOAD
    )
  )
);

Go sync from jobs_knowledge_base to isa632_7474656346303369.boopatt.jobindex

%md
### Testing the Index: Search for Products Similar to the Query

Before building the RAG pipeline, let's check if the index is ready. We will use `similarity_search` function to search for products that are similar to the query text.

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient(disable_notice=True)

question = "job"

try:
    # get index created in the previous step
    index = vsc.get_index("vs_endpoint_boopatt", "isa632_7474656346303369.boopatt.jobindex")
    
    # search for similar documents
    results = index.similarity_search(
        query_text = question,
        columns=["content"],
        num_results=4
    )

    # show the results
    docs = results.get("result", {}).get("data_array", [])
    print(docs)

except Exception as e:
    print(f"Error occurred while loading the index. Did you create an index in the previous step?: {e}")

%md
## Enable MLflow Tracing

Before we build our RAG pipeline, let's turn on MLflow tracing. This helps us automatically track and log what happens in our pipeline, making it easier to review and improve our work later. 

**💡 Note: MLflow logging is enabled by default, but we'll show how to enable it here for learning purposes.**

In [0]:
import mlflow
mlflow.langchain.autolog()
import os
os.environ["VS_INDEX_NAME"] = "isa632_7474656346303369.boopatt.jobindex"

%md
## Build a RAG Model

Now it's time to put everything together! We'll build a simple RAG (Retrieval-Augmented Generation) pipeline that uses the documents we found with Vector Search to help answer questions. This makes our model smarter by giving it extra context.

**⚠️ Notice:** You must pass the correct VS index name as environment variable. It is used in `agent.py` file to access the index.

In [0]:
# Load the agent code from the agent.py file. This approach allows you to modularize the agent logic and reuse it for deployment or further development.
# Reload agent.py to pick up the updated prompt, then test with verbose mode
import sys
import importlib

sys.path.insert(0, '/Workspace/Users/boopatt@miamioh.edu/Project')

if 'agent' in sys.modules:
    del sys.modules['agent']

from agent import AGENT

AGENT.agent.verbose = True

user_input = "what are the available roles for scala?"
request = {
    "input": [
        {"role": "user", "content": user_input}
    ]
}
resp = AGENT.predict(request)
print(resp)

%md
## Register the RAG Model to Unity Catalog

Once your RAG pipeline is working, you can register it in Unity Catalog. **This makes it easy to manage, share, and deploy your model later.** Registering your model is a best practice for keeping your work organized and ready for production.

In [0]:
import mlflow
from mlflow.models.resources import (
    DatabricksVectorSearchIndex
)
from mlflow.models import infer_signature
from pkg_resources import get_distribution

# Set Model Registry URI to Unity Catalog
mlflow.set_registry_uri("databricks-uc")

#model_name = f"{DA.catalog_name}.{DA.schema_name}.getstarted_genai_retrevial_demo"
model_name = "isa632_7474656346303369.boopatt.getstarted_job_listings" # replace boopatt with your MUID
vs_index_table_name="isa632_7474656346303369.boopatt.jobindex"

# Set the agent
mlflow.models.set_model(AGENT)

# Register the assembled RAG model in Model Registry with Unity Catalog
with mlflow.start_run(run_name="joblistings_demo") as run:
    model_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        pip_requirements=[
            f"langchain=={get_distribution('langchain').version}",
            f"databricks-vectorsearch=={get_distribution('databricks-vectorsearch').version}",
            f"databricks_langchain=={get_distribution('databricks_langchain').version}",
            f"mlflow=={get_distribution('mlflow').version}"
        ],
        resources=[
            DatabricksVectorSearchIndex(index_name=vs_index_table_name)
        ]
    )
model_uri = f"runs:/{run.info.run_id}/{model_info.name}"
model_version = mlflow.register_model(model_uri, model_name)
print(f"Model registered with name: {model_name} and version: {model_version.version}")

%md
## A. Serve the Model with Agent Framework

This section demonstrates how to serve a model from the model registry using Mosaic Model Serving and the Agent Framework. When models are served with the Agent Framework, a Review App is automatically deployed alongside the model, allowing you to interact with the model and gather human feedback on its responses.

In [0]:
import time
import mlflow
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointStateReady, EndpointStateConfigUpdate
from databricks import agents

# Deploy the model with the agent framework, passing VS_INDEX_NAME as an environment variable
deployment_info = agents.deploy(
    model_name, 
    model_version=1,
    scale_to_zero=True,
    environment_vars={"VS_INDEX_NAME": vs_index_table_name}
)

# Wait for the Review App and deployed model to be ready
w = WorkspaceClient()
print("\nWaiting for endpoint to deploy.  This can take 15 - 20 minutes.", end="")

while ((w.serving_endpoints.get(deployment_info.endpoint_name).state.ready == EndpointStateReady.NOT_READY) or (w.serving_endpoints.get(deployment_info.endpoint_name).state.config_update == EndpointStateConfigUpdate.IN_PROGRESS)):
    print(".", end="")
    time.sleep(30)

print("\nThe endpoint is ready!", end="")